## Report
```
The input DNA string (up to 1000 nt) is transferred to the GPU as a char array.
A CUDA kernel assigns one thread per nucleotide and each thread copies its character to an output array,
substituting 'U' wherever it finds 'T' and leaving all other bases unchanged.
The result is copied back to the host and printed as the transcribed RNA string.
```

In [1]:
%%writefile transcribing_dna_into_rna.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <cuda_runtime.h>

#define DNA_INPUT "GATGGAACTTGACTACGTAAATT"

__global__ void transcribing_dna_into_rna_kernel(const char *seq, char *out, int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n) {
        out[idx] = (seq[idx] == 'T') ? 'U' : seq[idx];
    }
}

void launch_rna_kernel(const char *h_seq, int n)
{
    char *d_seq = NULL;
    char *d_out = NULL;

    cudaMalloc((void **)&d_seq, n * sizeof(char));
    cudaMalloc((void **)&d_out, (n + 1) * sizeof(char));

    cudaMemset(d_out, 0, (n + 1) * sizeof(char));
    cudaMemcpy(d_seq, h_seq, n * sizeof(char), cudaMemcpyHostToDevice);

    int block_size = 256;
    int grid_size = (n + block_size - 1) / block_size;

    transcribing_dna_into_rna_kernel<<<grid_size, block_size>>>(d_seq, d_out, n);

    cudaDeviceSynchronize();

    char *h_out = (char *)malloc((n + 1) * sizeof(char));

    cudaMemcpy(h_out, d_out, (n + 1) * sizeof(char), cudaMemcpyDeviceToHost);

    h_out[n] = '\0';

    printf("%s\n", h_out);

    free(h_out);
    cudaFree(d_seq);
    cudaFree(d_out);
}

int main(void)
{
    const char *dna = DNA_INPUT;
    int n = (int)strlen(dna);

    launch_rna_kernel(dna, n);

    return 0;
}

Writing transcribing_dna_into_rna.cu


In [2]:
!nvcc -arch=sm_75 transcribing_dna_into_rna.cu -o transcribing_dna_into_rna

In [3]:
!./transcribing_dna_into_rna

GAUGGAACUUGACUACGUAAAUU
